# evaluation.ipynb

Zwei-Block-Evaluation fuer den KI-Copilot fuer Verzahnungswissen.

**Block 1** laedt alle PDFs aus `test_verzahnung/Input_Daten`, speichert die ausgelesenen Dokumente und erzeugt optional einmalig ein Evaluationsset mit ca. 100 Fragen. Jede Frage wird durch ein LLM gezielt zu einem `true_recursive_chunk` erzeugt.

**Block 2** fuehrt ein frei parametrisierbares Chunking und Dense/Hybrid-Retrieval aus und bewertet die Treffer gegen das Evaluationsset via Mean Reciprocal Rank.

Das Retrieval arbeitet ausschliesslich mit der Nutzerfrage. Die CAD-Bauteildaten fliessen im Gesamtsystem erst in der Antwortstufe (AnswerGenerator) als Kontext ein und beeinflussen damit nicht das Retrieval bzw. den hier gemessenen MRR. Diese Evaluation misst daher reine Retrieval-Qualitaet.

## Block 1: Input-Daten laden und Evaluationsset erzeugen/laden

In [1]:
# ============================================================
# BLOCK 1: DOKUMENTE LADEN + EVALUATIONSSET ERZEUGEN/LADEN
# ============================================================

import json
import random
import re
import sys
import time
from dataclasses import asdict
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "test_verzahnung":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.types import RawDocument, RawDocumentPage
from app.implementations.pdf_loader_pymupdf import PDFLoader
from app.implementations.chunker_recursive import RecursiveTextChunker
from app.implementations.text_split import approx_tokens
from app.implementations.ollama_client import OllamaClient

# ----------------------------
# Direkt modifizierbare Werte
# ----------------------------
input_data_dir = PROJECT_ROOT / "test_verzahnung" / "Input_Daten"
documents_cache_file = PROJECT_ROOT / "test_verzahnung" / "documents_cache.json"
evaluation_set_file = PROJECT_ROOT / "test_verzahnung" / "evaluation_set.json"

# True = Evaluationsset neu erzeugen. Danach auf False setzen, um das gespeicherte Set zu laden.
generate_evaluation_set = False

# Zielgroesse des Evaluationssets.
n_evaluation_questions = 100
random_seed = 42

# Initiales true-recursive Chunking fuer die Gold-Fragen.
eval_chunk_min = 80
eval_chunk_max = 512
eval_chunk_recursive_length = eval_chunk_max
eval_overlap_sentences = 1

# LLM fuer die Fragegenerierung.
ollama_url = "http://localhost:11434"
question_generation_llm_model = "llama3.2:3b"
ollama_timeout_s = 90
max_generation_retries = 3

# Optional: Chunks mit sehr wenig Fachinhalt vermeiden.
min_true_chunk_tokens_for_question = 80

input_data_dir.mkdir(parents=True, exist_ok=True)
print(f"Input-Ordner: {input_data_dir}")
print("Lege deine ca. 10 PDFs in diesen Ordner und fuehre Block 1 aus.")

def raw_document_to_dict(doc):
    return {
        "source_path": doc.source_path,
        "doc_hash": doc.doc_hash,
        "pages": [asdict(p) for p in doc.pages],
    }

def raw_document_from_dict(data):
    return RawDocument(
        source_path=data["source_path"],
        doc_hash=data["doc_hash"],
        pages=[RawDocumentPage(**p) for p in data["pages"]],
    )

def load_pdfs_and_cache():
    pdf_files = sorted(input_data_dir.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"Keine PDFs in {input_data_dir} gefunden.")
    loader = PDFLoader()
    docs = [loader.load(p) for p in pdf_files]
    documents_cache_file.write_text(
        json.dumps([raw_document_to_dict(d) for d in docs], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return docs

def load_cached_documents():
    if not documents_cache_file.exists():
        return load_pdfs_and_cache()
    data = json.loads(documents_cache_file.read_text(encoding="utf-8"))
    return [raw_document_from_dict(d) for d in data]

documents = load_pdfs_and_cache()
print(f"Geladene Dokumente: {len(documents)}")
for d in documents:
    tokens = sum(approx_tokens(p.text) for p in d.pages)
    print(f"- {Path(d.source_path).name}: {len(d.pages)} Seiten, ca. {tokens} Tokens, hash={d.doc_hash[:12]}")

def infer_gear_type_from_text(text):
    t = (text or "").casefold()
    if "schrägverzah" in t or "schraegverzah" in t:
        return "Schrägverzahnung"
    if "stirnrad" in t or "geradverzah" in t:
        return "Stirnrad"
    if "kegelrad" in t:
        return "Kegelrad"
    if "schnecken" in t:
        return "Schneckenrad"
    if "innenverzah" in t:
        return "Innenverzahnung"
    return ""

def build_true_recursive_chunks():
    chunker = RecursiveTextChunker(
        min_chunk_tokens=eval_chunk_min,
        max_chunk_tokens=eval_chunk_recursive_length,
        overlap_sentences=eval_overlap_sentences,
    )
    chunks = []
    for doc in documents:
        chunks.extend(chunker.chunk(doc))
    return [c for c in chunks if approx_tokens(c.text) >= min_true_chunk_tokens_for_question]

def make_question_prompt(chunk_text):
    return f"""
Du erzeugst eine Evaluationsfrage fuer ein Retrieval-System im Bereich Verzahnungswissen.

Ziel:
- Erzeuge genau eine konkrete deutsche Frage.
- Die Frage muss spezifisch durch den folgenden Text-Chunk beantwortbar sein.
- Die Frage soll nicht wortgleich einen ganzen Satz aus dem Chunk kopieren.
- Die Frage darf kein Wissen voraussetzen, das nicht im Chunk steht.
- Keine Antwort erzeugen, nur JSON.

Gib ausschliesslich dieses JSON-Format zurueck:
{{"question": "...", "answer_hint": "kurzer Hinweis, welche Information im Chunk relevant ist"}}

TEXT-CHUNK:
{chunk_text[:3500]}
""".strip()

def generate_question_for_chunk(client, chunk):
    prompt = make_question_prompt(chunk.text)
    last_error = None
    for _ in range(max_generation_retries):
        try:
            data = client.generate_json(
                model=question_generation_llm_model,
                prompt=prompt,
                temperature=0.2,
                max_tokens=180,
            )
            question = str(data.get("question", "")).strip()
            answer_hint = str(data.get("answer_hint", "")).strip()
            if question.endswith("?") and len(question) >= 20:
                return question, answer_hint
        except Exception as exc:
            last_error = exc
            time.sleep(1)
    raise RuntimeError(f"Frage konnte nicht erzeugt werden: {last_error}")

if generate_evaluation_set:
    true_chunks = build_true_recursive_chunks()
    if len(true_chunks) < n_evaluation_questions:
        print(f"Warnung: Nur {len(true_chunks)} geeignete Chunks fuer {n_evaluation_questions} Fragen gefunden.")
    rng = random.Random(random_seed)
    selected_chunks = rng.sample(true_chunks, k=min(n_evaluation_questions, len(true_chunks)))
    client = OllamaClient(base_url=ollama_url, timeout_s=ollama_timeout_s)
    evaluation_items = []
    for idx, chunk in enumerate(selected_chunks, start=1):
        print(f"[{idx}/{len(selected_chunks)}] Frage generieren: {Path(chunk.source_path).name}, Seite {chunk.page_number}, Chunk {chunk.position}", flush=True)
        question, answer_hint = generate_question_for_chunk(client, chunk)
        evaluation_items.append({
            "id": f"eval_{idx:03d}",
            "question": question,
            "answer_hint": answer_hint,
            "true_recursive_chunk": chunk.text,
            "true_source_path": chunk.source_path,
            "true_source_name": Path(chunk.source_path).name,
            "true_page_number": chunk.page_number,
            "true_position": chunk.position,
            "true_doc_hash": chunk.doc_hash,
            "true_gear_type": infer_gear_type_from_text(chunk.text),
            "true_chunk_tokens": approx_tokens(chunk.text),
        })
    payload = {
        "created_at_unix": int(time.time()),
        "generator_model": question_generation_llm_model,
        "input_data_dir": str(input_data_dir),
        "true_chunking": {
            "method": "recursive",
            "chunk_min": eval_chunk_min,
            "chunk_max": eval_chunk_recursive_length,
            "overlap_sentences": eval_overlap_sentences,
        },
        "items": evaluation_items,
    }
    evaluation_set_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Evaluationsset gespeichert: {evaluation_set_file}")
else:
    if not evaluation_set_file.exists():
        raise FileNotFoundError(
            f"{evaluation_set_file} existiert noch nicht. Setze generate_evaluation_set=True und fuehre Block 1 einmal aus."
        )
    payload = json.loads(evaluation_set_file.read_text(encoding="utf-8"))
    evaluation_items = payload["items"]
    print(f"Bestehendes Evaluationsset geladen: {evaluation_set_file}")

print(f"Evaluationsfragen: {len(evaluation_items)}")
for item in evaluation_items[:5]:
    print(f"- {item['id']}: {item['question']} | true={item.get('true_source_name')} S.{item.get('true_page_number')}")

Input-Ordner: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/Input_Daten
Lege deine ca. 10 PDFs in diesen Ordner und fuehre Block 1 aus.
Geladene Dokumente: 2
- Manufacturing_of_Gears.pdf: 23 Seiten, ca. 11330 Tokens, hash=21ffc36d0cc0
- gear_skiving_technology.pdf: 18 Seiten, ca. 15774 Tokens, hash=1089d8e53363
Bestehendes Evaluationsset geladen: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/evaluation_set.json
Evaluationsfragen: 73
- eval_001: Wie können Tooth-Maschinen mit einer Universal-Tool wie einem Endmull für die Multipass-Methode verwendet werden? | true=Manufacturing_of_Gears.pdf S.10
- eval_002: Was die Bezeichnung "Involute" in der Zahntechnik und ihre Eigenschaften genau beschrieben? | true=Manufacturing_of_Gears.pdf S.2
- eval_003: Wann wurde die Gear Skiving-Technologie von Kojima et al. erstentwickelt? | true=gear_skiving_technology.pdf S.1
- eval_004: Was in der Publikation 'Machining Gears with an I

## Block 2: Modell konfigurieren, Retrieval ausfuehren und MRR berechnen

Pipeline pro Frage: `Nutzerfrage → Embedding → Hybrid-Retrieval`

$$\mathrm{MRR}=\frac{1}{|Q|}\sum_{i=1}^{|Q|}\frac{1}{\mathrm{rank}_i}$$

Dabei ist $|Q|$ die Anzahl der Evaluationsfragen und $\mathrm{rank}_i$ der Rang des ersten relevanten Treffers fuer Frage $i$. Wird kein relevanter Treffer gefunden, zaehlt der Beitrag als $0$.

Das Retrieval nutzt ausschliesslich die Nutzerfrage. Die CAD-Bauteildaten gehoeren im Gesamtsystem in die Antwortstufe und beeinflussen das Retrieval nicht.

In [2]:
# ============================================================
# BLOCK 2: MODELL + RETRIEVAL + MRR
# ============================================================

import json
import re
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "test_verzahnung":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.types import RawDocument, RawDocumentPage
from app.implementations.chunker_recursive import RecursiveTextChunker
from app.implementations.chunker_semantic import SemanticChunker
from app.implementations.embedder_bge_m3 import BGEM3Embedder

# ----------------------------
# Direkt modifizierbare Werte
# ----------------------------
documents_cache_file = PROJECT_ROOT / "test_verzahnung" / "documents_cache.json"
evaluation_set_file = PROJECT_ROOT / "test_verzahnung" / "evaluation_set.json"
results_file = PROJECT_ROOT / "test_verzahnung" / "evaluation_results" / f"evaluation_results_{int(time.time())}.json"

# Chunking im zu evaluierenden Modell.
# chunk_method: "recursive" oder "chunker". "chunker" bedeutet hier SemanticChunker.
chunk_method = "recursive"
chunk_min = 80
chunk_max = 512
chunk_recursive_length = chunk_max
threshold_semantic_chunker = 0.40
overlap_sentences = 1

# Embedding + Hybrid-Retrieval (entspricht dem HybridRetriever des Hauptmodells).
embedding_model = "BAAI/bge-m3"
embedding_device = "mps"  # "mps", "cpu" oder "cuda"
embedding_max_length = 8192

use_hybrid = True
hybrid_dense_weight = 0.70
hybrid_sparse_weight = 1 - hybrid_dense_weight

cos_score_min_threshold = 0.4
top_k_chunks_threshold = 10

# Vergleich zwischen retrieved chunk und true_recursive_chunk.
# Bei anderem Chunking kann der Original-Chunk nicht per ID gematcht werden; daher Text-Overlap.
true_chunk_overlap_threshold = 0.35

# Weitere wichtige Variablen.
deduplicate_retrieved_chunks = True
random_seed = 42

def raw_document_from_dict(data):
    return RawDocument(
        source_path=data["source_path"],
        doc_hash=data["doc_hash"],
        pages=[RawDocumentPage(**p) for p in data["pages"]],
    )

if not documents_cache_file.exists():
    raise FileNotFoundError("documents_cache.json fehlt. Fuehre zuerst Block 1 aus.")
if not evaluation_set_file.exists():
    raise FileNotFoundError("evaluation_set.json fehlt. Fuehre zuerst Block 1 mit generate_evaluation_set=True aus.")

documents = [raw_document_from_dict(d) for d in json.loads(documents_cache_file.read_text(encoding="utf-8"))]
evaluation_items = json.loads(evaluation_set_file.read_text(encoding="utf-8"))["items"]
print(f"Dokumente: {len(documents)} | Evaluationsfragen: {len(evaluation_items)}")

def build_chunker(embedder_for_semantic=None):
    if chunk_method == "recursive":
        return RecursiveTextChunker(
            min_chunk_tokens=chunk_min,
            max_chunk_tokens=chunk_recursive_length,
            overlap_sentences=overlap_sentences,
        )
    if chunk_method == "chunker":
        if embedder_for_semantic is None:
            raise ValueError("chunk_method='chunker' braucht den Embedder fuer SemanticChunker.")
        return SemanticChunker(
            embedder=embedder_for_semantic,
            threshold=threshold_semantic_chunker,
            min_chunk_tokens=chunk_min,
            max_chunk_tokens=chunk_max,
            overlap_sentences=overlap_sentences,
        )
    raise ValueError("chunk_method muss 'recursive' oder 'chunker' sein.")

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))

def sparse_dot(a, b):
    if not a or not b:
        return 0.0
    if len(a) > len(b):
        a, b = b, a
    return sum(float(v) * float(b.get(k, 0.0)) for k, v in a.items())

token_re = re.compile(r"[A-Za-zÄÖÜäöüß0-9_+-]+")
def token_set(text):
    return {t.casefold() for t in token_re.findall(text or "") if len(t) > 1}

def overlap_score(retrieved_text, true_text):
    rt = token_set(retrieved_text)
    tt = token_set(true_text)
    if not rt or not tt:
        return 0.0
    # Recall-artiger Overlap: Wie viel vom True-Chunk ist im Treffer enthalten?
    return len(rt & tt) / len(tt)

def normalize_text(text):
    return re.sub(r"\s+", " ", (text or "").strip().casefold())

print("Lade Embedder...")
embedder = BGEM3Embedder(
    model_name=embedding_model,
    device=embedding_device,
    max_length=embedding_max_length,
    use_sparse=use_hybrid,
)

print("Chunking...")
chunker = build_chunker(embedder if chunk_method == "chunker" else None)
chunks = []
for doc in documents:
    chunks.extend(chunker.chunk(doc))
print(f"Chunks im Modelllauf: {len(chunks)}")

print("Embeddings fuer Chunks berechnen...")
chunk_embedding_result = embedder.embed([c.text for c in chunks])
chunk_dense_vectors = chunk_embedding_result.dense_vectors
chunk_sparse_vectors = chunk_embedding_result.sparse_vectors or [{} for _ in chunks]

def retrieve_for_query(query):
    q_embedding = embedder.embed([query])
    q_dense = q_embedding.dense_vectors[0]
    q_sparse = q_embedding.sparse_vectors[0] if q_embedding.sparse_vectors else {}

    scored = []
    for chunk, dense_vec, sparse_vec in zip(chunks, chunk_dense_vectors, chunk_sparse_vectors):
        dense_score = cosine(q_dense, dense_vec)
        if dense_score < cos_score_min_threshold:
            continue
        sparse_score = sparse_dot(q_sparse, sparse_vec) if use_hybrid else 0.0
        final_score = dense_score
        if use_hybrid:
            final_score = hybrid_dense_weight * dense_score + hybrid_sparse_weight * sparse_score
        scored.append({
            "chunk": chunk,
            "dense_score": dense_score,
            "sparse_score": sparse_score,
            "score": final_score,
        })

    scored.sort(key=lambda x: x["score"], reverse=True)

    if deduplicate_retrieved_chunks:
        deduped = []
        seen = set()
        for row in scored:
            key = normalize_text(row["chunk"].text)
            if key in seen:
                continue
            seen.add(key)
            deduped.append(row)
        scored = deduped

    return scored[:top_k_chunks_threshold]

def reciprocal_rank_for_item(item, retrieved):
    true_text = item["true_recursive_chunk"]
    best_overlap = 0.0
    best_rank = None
    for rank, row in enumerate(retrieved, start=1):
        ov = overlap_score(row["chunk"].text, true_text)
        best_overlap = max(best_overlap, ov)
        if ov >= true_chunk_overlap_threshold:
            best_rank = rank
            break
    return (1.0 / best_rank if best_rank else 0.0), best_rank, best_overlap

print("Retrieval-Evaluation startet...")
rows = []
reciprocal_ranks = []
for idx, item in enumerate(evaluation_items, start=1):
    retrieved = retrieve_for_query(item["question"])
    rr, rank, best_overlap = reciprocal_rank_for_item(item, retrieved)
    reciprocal_ranks.append(rr)
    rows.append({
        "id": item["id"],
        "question": item["question"],
        "reciprocal_rank": rr,
        "rank": rank,
        "best_overlap": best_overlap,
        "retrieved_count": len(retrieved),
        "true_source_name": item.get("true_source_name"),
        "true_page_number": item.get("true_page_number"),
        "top1_source": Path(retrieved[0]["chunk"].source_path).name if retrieved else None,
        "top1_page": retrieved[0]["chunk"].page_number if retrieved else None,
        "top1_score": retrieved[0]["score"] if retrieved else None,
        "top1_dense_score": retrieved[0]["dense_score"] if retrieved else None,
        "top1_sparse_score": retrieved[0]["sparse_score"] if retrieved else None,
    })
    if idx % 10 == 0 or idx == len(evaluation_items):
        print(f"  {idx}/{len(evaluation_items)} Fragen evaluiert", flush=True)

mrr_score = sum(reciprocal_ranks) / max(1, len(reciprocal_ranks))
hit_at_k = sum(1 for r in rows if r["rank"] is not None) / max(1, len(rows))

print("\n==============================")
print(f"MRR Score: {mrr_score:.4f}")
print(f"Hit@{top_k_chunks_threshold}: {hit_at_k:.4f}")
print(f"Fragen: {len(rows)}")
print("==============================\n")

failed = [r for r in rows if r["rank"] is None]
print(f"Nicht gefundene True-Chunks: {len(failed)}")
for r in failed[:10]:
    print(f"- {r['id']} | overlap={r['best_overlap']:.2f} | {r['question']}")

result_payload = {
    "mrr_score": mrr_score,
    "hit_at_k": hit_at_k,
    "rows": rows,
    "settings": {
        "chunk_method": chunk_method,
        "chunk_min": chunk_min,
        "chunk_max": chunk_max,
        "chunk_recursive_length": chunk_recursive_length,
        "threshold_semantic_chunker": threshold_semantic_chunker,
        "overlap_sentences": overlap_sentences,
        "embedding_model": embedding_model,
        "embedding_device": embedding_device,
        "use_hybrid": use_hybrid,
        "hybrid_dense_weight": hybrid_dense_weight,
        "hybrid_sparse_weight": hybrid_sparse_weight,
        "cos_score_min_threshold": cos_score_min_threshold,
        "top_k_chunks_threshold": top_k_chunks_threshold,
        "true_chunk_overlap_threshold": true_chunk_overlap_threshold,
    },
}
results_file.write_text(json.dumps(result_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Ergebnisse gespeichert: {results_file}")

/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Dokumente: 2 | Evaluationsfragen: 73
Lade Embedder...
Chunking...
Chunks im Modelllauf: 73
Embeddings fuer Chunks berechnen...
Retrieval-Evaluation startet...
  10/73 Fragen evaluiert
  20/73 Fragen evaluiert
  30/73 Fragen evaluiert
  40/73 Fragen evaluiert
  50/73 Fragen evaluiert
  60/73 Fragen evaluiert
  70/73 Fragen evaluiert
  73/73 Fragen evaluiert

MRR Score: 0.6755
Hit@10: 0.9041
Fragen: 73

Nicht gefundene True-Chunks: 7
- eval_013 | overlap=0.25 | Wie kann die Machbarkeit verschiedener Schneidverfahren für Zahnräder bestimmt werden?
- eval_020 | overlap=0.33 | Welche Rolle spielen die Schneidewinkel Ö und η in der Verzahnung?
- eval_022 | overlap=0.31 | Welche Rolle spielt die Schmälerung bei der Simulation von Zahnradschmiedearbeiten?
- eval_026 | overlap=0.33 | Was ist ein bestimmtes Werkzeug für die Skiving von Zahnrädern?
- eval_051 | overlap=0.32 | Wie viele Hauptbereiche hat die Forschung auf dem Gebiet der Zahnradbearbeitung identifiziert?
- eval_052 | overlap=0.34 |